# Master Setup Notebook

Run this notebook to create the complete folder structure and configuration files to run the Eutrophication Workbench workflow.

---

## Instructions

1. Execute the Python cell below to set up the directories and copy necessary files.
2. Enter the required inputs when prompted (user initials, years, depths).
3. The script will create all folders and generate configuration files automatically.

---

## Folder Structure Created

- `1_DatalakeQuery/`: For BEACON queries on merged BEACON instanses using jupyter notebooks
- `2_CWduplicates-tool/`: Configuration for duplicate removal tool
- `3_FileForge/`: FileForge configurations and outputs
- `4_webODV/`: ODV data in webODV

---

> **Note:** Ensure you have access to the source directories for copying files. If not, warnings will be printed.

In [ ]:
from pathlib import Path
from datetime import datetime
from shutil import copy2
import os

root = Path.cwd()

user = input("Enter your initials: ").strip().replace(" ", "_")
if not user:
    user = "anonymous"

year_from = input("Enter start year (YYYY): ").strip()
year_to = input("Enter end year (YYYY): ").strip()
depth_from = input("Enter depth from (m): ").strip()
depth_to = input("Enter depth to (m): ").strip()

if not year_from:
    year_from = "2015"
if not year_to:
    year_to = "2020"
if not depth_from:
    depth_from = "0"
if not depth_to:
    depth_to = "100"

date = datetime.now().strftime("%Y%m%d")
output_base = root / "1_DatalakeQuery" / "outputs" / f"{date}_{user}"

dirs = [
    root / "1_DatalakeQuery" / "Merged_notebooks",
    output_base / "in_raw" / "EUTRO_CMEMS",
    output_base / "in_raw" / "EUTRO_EC",
    output_base / "in_raw" / "EUTRO_WOD",
    root / "2_CWduplicates-tool" / "Configuration_files" / "Config_Profiles",
    root / "2_CWduplicates-tool" / "Configuration_files" / "Config_Timeseries",
    root / "2_CWduplicates-tool" / "Configuration_files" / "Results",
    root / "2_CWduplicates-tool" / "Configuration_files" / "cache_empty",
    root / "3_FileForge"  / "Configuration_files",
    root / "3_FileForge"  / "outputs",
    root / "4_webODV" / "collections",
]

for path in dirs:
    path.mkdir(parents=True, exist_ok=True)
    
# Copy beacon merged notebook
src_nb = Path(os.path.expanduser("~/blue-cloud-dataspace/Eutrophication_Workbench/Beacon_Notebooks/1.5.4/01_EWB_BEACON=merged-v1.5.4_FILTER=BDI-FeatureType_VAR=allEOV_OUT=parquet.ipynb"))
dst_nb = root / "1_DatalakeQuery" / "Merged_notebooks" / src_nb.name
dst_nb.parent.mkdir(parents=True, exist_ok=True)
copy2(src_nb, dst_nb)


# Copy FileForge configuration files
src_config = Path(os.path.expanduser("~/workspace/VREFolders//Eutrophication-Workbench/3_FileForge/"))
dst_config = root / "3_FileForge" / "Configuration_files"

if src_config.exists():
    for config_file in src_config.glob("*"):
        if config_file.is_file():
            copy2(config_file, dst_config / config_file.name)
else:
    print(f"Warning: Source config directory not found at {src_config}")

cw_profile_template = f"""default :
  operator :
    name : "{user}"
  output :
    filename : "02_BC_EWB_CW_PR_{depth_from}-{depth_to}_{year_from}-{year_to}"
  #-----------------------------
  # SOURCE INFORMATION
  #-------------------------
  sources :
    - src_short_name : "BEACON_CMEMS_BGC"
      raw_data :
        flag : 1
        reader_type : "BEACON_PARQUET"
        data_dir : "EUTRO_CMEMS"
      trustlevel : 1

    - src_short_name : "BEACON_EC"
      raw_data :
        flag : 1
        reader_type : "BEACON_PARQUET"
        data_dir : "EUTRO_EC"
      trustlevel : 2

    - src_short_name : "BEACON_WOD"
      raw_data :
        flag : 1
        reader_type : "BEACON_PARQUET"
        data_dir : "EUTRO_WOD"
      trustlevel : 3

  #--------------------------
  #DATA SELECTION
  #------------------------------
  data_sel :
    - region : NULL
    - instr : NULL
    - groupParam : NULL
    - dataType : ["TRP", "PR", "TSP"]
    - period : NULL
  dupli_def :
    time_unit : "minutes"
    time_diff : 15
    pos_diff : 15
    instr_option : 2
    src_option : 2
    aggr_option : 4
    graph_flag : 1
  removal :
    flag : 1
    group : 0
    rules_to_apply : [2,3,4,5]
"""

profile_path = (root / "2_CWduplicates-tool" / "Configuration_files" / "Config_Profiles" / "cw_user_config.yaml")
profile_path.write_text(cw_profile_template, encoding="utf-8")


print("Created master folder structure under:", root)